# Figure 3 — coincidence and suppression

The simulation cells reproduce the plotted panels. Set `RUN_EXPENSIVE = True` to refresh the saved chirp grid; the default loads its released results.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Pulse-number responses (A–C) and example traces (D–E)

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import plotting as p
from stim_gen import makeCHIRPstim
from models.fix import (Params, Params_noada_AN1, Params_noada_LN3,
                        Params_noInh_in_AN1, Params_noInh_from_LN2, run_model)

stim, song, _, _ = makeCHIRPstim(
    cdur=np.arange(0, 330, 40),
    cpau=320 - np.arange(0, 330, 40),
    ppau=20, pdur=20, Fs=1000, num_steps=32,
)
stim = jnp.array(stim)
full = run_model(stim, Params(), verbose_output=True, return_dict=True)
no_coincidence = Params()
no_coincidence.an1_ln3_gain = 0
without_coincidence = run_model(stim, no_coincidence, verbose_output=False)
neurons = ['an1', 'ln2', 'ln5', 'ln3', 'ln4']
colors = ['#176c9c', '#55aadd', '#009a7a', '#d66000', '#cb77a5']
tuning = np.array([np.nanmean(full[key], axis=0) for key in neurons]).T
tuning_no_coincidence = np.nanmean(np.stack(without_coincidence), axis=1).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, responses, title in zip(
    axes[:2], [tuning, tuning_no_coincidence],
    ['Full model', 'No coincidence'],
):
    for index, (name, color) in enumerate(zip(neurons, colors)):
        ax.plot(range(5), responses[:5, index] / responses[1, index],
                'o-', color=color, label=name.upper())
    ax.plot([0, 4], [0, 4], color='black', lw=1)
    ax.set(xlim=(-.2, 4.2), ylim=(0, 4), xticks=range(5),
           xlabel='Number of pulses', title=title)
axes[0].set_ylabel('Response / one-pulse response')
axes[0].legend(frameon=False, handlelength=0, labelcolor='linecolor')
for name, response, color in [
    ('Full model', tuning, '0.3'),
    ('No coincidence', tuning_no_coincidence, '#d66000'),
]:
    axes[2].plot(range(5), response[2] / (2 * response[1]),
                 'o-', color=color, label=name)
axes[2].axhline(1, color='black', lw=1)
axes[2].set(xlim=(-.2, 4.2), xticks=range(5),
            xticklabels=[name.upper() for name in neurons],
            ylabel='Two-pulse / linear prediction')
axes[2].legend(frameon=False, handlelength=0, labelcolor='linecolor')
fig.tight_layout()
fig.savefig(FIGURES / 'fig_3A-C.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(7.5, 5), sharex=True)
window = slice(350, 600)
for column, stimulus_index in enumerate([1, 2]):
    for row, signal in enumerate(['stim', 'an1', 'ln5', 'ln3', 'ln4']):
        ax = axes[row, column]
        ax.plot(np.arange(window.stop - window.start),
                full[signal][window, stimulus_index], color='black')
        ax.set_yticks([])
        ax.spines[['top', 'right', 'left']].set_visible(False)
        if column == 0:
            ax.set_ylabel(signal.upper(), rotation=0, labelpad=20)
        if row == 0:
            ax.set_title(f'{"D" if column == 0 else "E"}: {column + 1} pulse{"s" if column else ""}')
axes[-1, 0].set_xlabel('Time [ms]')
axes[-1, 1].set_xlabel('Time [ms]')
fig.tight_layout()
fig.savefig(FIGURES / 'fig_3D-E.pdf', bbox_inches='tight')
plt.show()

## Chirp suppression (G–I, K–M)

In [ ]:
RUN_EXPENSIVE = False
CACHE = (ROOT / 'results' if RUN_EXPENSIVE else SAVED_RESULTS) / 'fig3_suppression.npz'
variants = {
    'full': Params,
    'noada_AN1': Params_noada_AN1,
    'noada_LN3': Params_noada_LN3,
    'noInh_AN1': Params_noInh_in_AN1,
    'noinh_LN4': Params_noInh_from_LN2,
}
if RUN_EXPENSIVE:
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    grid_stim, grid_song, _, _ = makeCHIRPstim(
        cpauMax=1020, cdurMax=1020, cpauMin=20, cdurMin=20,
        ppau=20, pdur=20, Fs=1000, num_steps=32, stim_len=10_000,
    )
    grid_stim = jnp.array(grid_stim)
    saved = {f's_{key}': np.asarray(value) for key, value in grid_song.items()
             if key != 'stim'}
    for key, make_params in variants.items():
        output = run_model(grid_stim, make_params(),
                           verbose_output=False, return_dict=True)
        saved[f's_rXY_{key}'] = np.nanmean(output['ln4'], axis=0)
        if key in ('full', 'noada_AN1', 'noInh_AN1'):
            for signal in ('an1', 'ln4'):
                saved[f'trace_{key}_{signal}'] = np.asarray(output[signal][:1100])
    np.savez_compressed(CACHE, **saved)
with np.load(CACHE) as saved:
    grid_song = {key[2:]: saved[key] for key in saved.files if key.startswith('s_')}
    traces = {
        key: {signal: saved[f'trace_{key}_{signal}']
              for signal in ('an1', 'ln4')}
        for key in ('full', 'noada_AN1', 'noInh_AN1')
    }

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 5), sharey=True)
comparisons = [
    [('full', 'black', 'Full model'),
     ('noada_AN1', '#176c9c', 'No div. norm. in AN1'),
     ('noada_LN3', '#d66000', 'No div. norm. in LN3')],
    [('full', 'black', 'Full model'),
     ('noInh_AN1', '#176c9c', 'No AN1 inhibitory filter'),
     ('noinh_LN4', '#cb77a5', 'No inhibition in LN4')],
]
for row, (filter_key, window, x_key, xlabel, scale) in enumerate([
    ('CPER', (240, 320), 'CDC', 'Chirp duty cycle', 1),
    ('CDC', (.4, .6), 'CPER', 'Chirp period [s]', 1000),
]):
    for column, lines in enumerate(comparisons):
        ax = axes[row, column]
        for key, color, label in lines:
            x, _, smooth = p.transect_est(grid_song, filter_key, window,
                                          x_key, f'rXY_{key}')
            ax.plot(x / scale, p.nmax(smooth), color=color, label=label)
        ax.set(xlim=(0, 1), ylim=(0, 1.05), xlabel=xlabel)
        if column == 0:
            ax.set_ylabel('Phonotaxis')
        if row == 0:
            ax.legend(frameon=False, handlelength=1, fontsize=8)
        ax.set_title([['G', 'H'], ['K', 'L']][row][column], loc='left')
fig.tight_layout()
fig.savefig(FIGURES / 'fig_3G-H_K-L.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True)
for column, (panel, indices, variant, title) in enumerate([
    ('I', [556, 495, 211], 'noada_AN1', 'No AN1 divisive normalization'),
    ('M', [297, 561, 825], 'noInh_AN1', 'No AN1 inhibitory filter'),
]):
    for row, signal in enumerate(('an1', 'ln4')):
        ax = axes[row, column]
        for offset, stimulus_index in enumerate(indices):
            shift = (5 if signal == 'an1' else 10) * offset
            ax.plot(traces['full'][signal][:, stimulus_index] + shift,
                    color='0.4', alpha=.8, label='Full' if offset == 0 else None)
            ax.plot(traces[variant][signal][:, stimulus_index] + shift,
                    color='#176c9c', alpha=.8,
                    label=title if offset == 0 else None)
        ax.set_yticks([])
        ax.spines[['top', 'right', 'left']].set_visible(False)
        if row == 0:
            ax.set_title(f'{panel}: {title}')
            ax.legend(frameon=False, fontsize=8)
        if column == 0:
            ax.set_ylabel(signal.upper(), rotation=0, labelpad=20)
    axes[1, column].set_xlabel('Time [ms]')
fig.tight_layout()
fig.savefig(FIGURES / 'fig_3I_M.pdf', bbox_inches='tight')
plt.show()